# Model Validation with Real Data
### A beginner-friendly Google Colab lab

Learn **data splits**, **overfitting and underfitting**, the **bias–variance tradeoff**, and **cross-validation** with actual observations, rather than generated data.

**Main dataset:** scikit-learn's diabetes regression dataset: baseline measurements from 442 patients and a numerical measure of disease progression one year later. We predict that numerical score, not whether a person has diabetes. This is a classroom prediction exercise.

**A second dataset:** the Wisconsin Diagnostic Breast Cancer dataset supplies real class labels for stratified-splitting examples.

Both datasets are bundled with scikit-learn. **No files, API keys, or dataset downloads are needed.** Random numbers below only choose or resample existing rows; they never create input measurements or targets.

**In Colab:** choose **File → Upload notebook**, then **Runtime → Run all**. A CPU is enough. Run cells in order; change the clearly marked settings to explore. Saved outputs show a complete run. Plotly charts render interactively in Colab; rerun a chart cell if its saved output is hidden. Dataset loading itself needs no downloads.

**Before you start:** familiarity with Python variables and lists helps. All machine-learning terms are explained as they appear.

## Learning journey
1. Data splits: learn, choose, and audit.
2. Overfitting and underfitting: compare simple and flexible prediction rules.
3. Bias–variance tradeoff: retrain on different samples and inspect stability.
4. Cross-validation: compare settings across several rehearsals, then test once.

**Color key:** teal = training; orange diamonds/dashed lines = validation; purple = final test. Lower error is better.

### One repeated coding pattern
Every model example follows **create → fit → predict → score**. Model settings control how the rule is learned; chart settings only change its display. Read the explanation, predict the outcome, run the cell, and discuss the chart before moving on.

**Beginner route:** follow the examples and ordinary K-fold loop first. Bootstrap bookkeeping, GridSearchCV, and custom group/time helper functions are marked as optional extensions; their outputs and purpose matter more than memorizing their syntax.


## Setup
- **NumPy** works with arrays of numbers.
- **pandas** displays tables.
- **Plotly** draws interactive charts. Hover for values, drag to zoom, double-click to reset, and click legend items to hide or show traces.
- **scikit-learn** supplies the datasets, models, and validation tools.

A **feature** is an input, such as BMI. A **target** is what we predict. `X` contains feature rows and `y` contains targets. `X` stays two-dimensional even with one feature.

In [1]:
import numpy as np
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots

from sklearn.datasets import load_diabetes, load_breast_cancer
from sklearn.tree import DecisionTreeRegressor
from sklearn.dummy import DummyRegressor
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler, PolynomialFeatures
from sklearn.base import clone
from sklearn.metrics import mean_squared_error
from sklearn.model_selection import (
    train_test_split, KFold, StratifiedKFold, GroupKFold,
    TimeSeriesSplit, cross_val_score, GridSearchCV
)

SEED = 42
TRAIN_COLOR = '#087e8b'
VAL_COLOR = '#bc4b12'
TEST_COLOR = '#7255b8'
MODEL_COLOR = '#384b63'

# Raw features: no scaling based on the whole dataset.
diabetes = load_diabetes(as_frame=True, scaled=False)
features = diabetes.data
features.columns = features.columns.str.lower()
target = diabetes.target
print('Feature table:', features.shape)
print('Target: a numerical progression score, not a yes/no diagnosis.')
print(features.head().round(2).to_string())
print('First target values:', target.head().to_list())

Feature table: (442, 10)
Target: a numerical progression score, not a yes/no diagnosis.
    age  sex   bmi     bp     s1     s2    s3   s4    s5    s6
0  59.0  2.0  32.1  101.0  157.0   93.2  38.0  4.0  4.86  87.0
1  48.0  1.0  21.6   87.0  183.0  103.2  70.0  3.0  3.89  69.0
2  72.0  2.0  30.5   93.0  156.0   93.6  41.0  4.0  4.67  85.0
3  24.0  1.0  25.3   84.0  198.0  131.4  40.0  5.0  4.89  89.0
4  50.0  1.0  23.0  101.0  192.0  125.4  52.0  4.0  4.29  80.0
First target values: [151.0, 75.0, 141.0, 206.0, 135.0]


### Reusable chart helpers: display only
These two small functions avoid repeating chart formatting. They **do not train models or calculate scores**. `show_chart` displays a Plotly figure; `add_patient_points` puts the same training and validation patients into a comparison panel.

In a subplot, `row` and `col` identify the panel. `go.Scatter` draws dots or lines; `go.Bar` draws bars. Chart options do not affect model fitting. You can leave this cell unchanged and focus on the modeling cells.

In [2]:
def show_chart(fig, title, height=450):
    fig.update_layout(
        title=title, template='plotly_white', height=height,
        legend=dict(orientation='h', y=-0.20),
        margin=dict(t=100, b=110)
    )
    fig.show(config={'displaylogo': False, 'scrollZoom': True})


def add_patient_points(fig, row=1, col=1, show_legend=False):
    fig.add_trace(go.Scatter(
        x=X_train[:, 0], y=y_train, mode='markers',
        name='Training patients', legendgroup='training', showlegend=show_legend,
        marker=dict(color=TRAIN_COLOR, size=5, opacity=0.35)
    ), row=row, col=col)
    fig.add_trace(go.Scatter(
        x=X_val[:, 0], y=y_val, mode='markers',
        name='Validation patients', legendgroup='validation', showlegend=show_legend,
        marker=dict(color=VAL_COLOR, symbol='diamond', size=6, opacity=0.5)
    ), row=row, col=col)

# 1. Data splits

## A textbook, a practice exam, and a sealed final exam
**Training** teaches a model a rule. **Validation** helps choose its settings. **Test** audits the chosen approach at the end.

**Generalization** means predicting new, relevant examples well. A model can memorize familiar examples and still generalize poorly.

| Set | Purpose | Allowed use |
|---|---|---|
| Training | Learn a rule | Fit the model and learned preprocessing |
| Validation | Choose settings | Compare models or hyperparameters |
| Test | Final independent audit | Score the final choice after selection |

To make curves easy to draw, Sections 1–3 use **one real feature: BMI**. Section 4 uses **all 10 features**. A one-feature model deliberately ignores useful information; do not interpret it as a strong prediction system.

We split **row IDs first**, so the same patient assignments can later be reused with all features.

In [3]:
# SETTINGS: two sequential splits give approximately 60/20/20.
# Whole patient counts mean the percentages are not perfectly exact.
row_ids = np.arange(len(features))
dev_ids, test_ids = train_test_split(
    row_ids, test_size=0.20, random_state=SEED
)
train_ids, val_ids = train_test_split(
    dev_ids, test_size=0.25, random_state=SEED
)

X_bmi = features[['bmi']].to_numpy()  # Double brackets keep X two-dimensional.
y = target.to_numpy()
X_train, y_train = X_bmi[train_ids], y[train_ids]
X_val, y_val = X_bmi[val_ids], y[val_ids]
# Only the test IDs are needed until the final audit.

split_table = pd.DataFrame({
    'Set': ['Training', 'Validation', 'Test'],
    'Patients': [len(train_ids), len(val_ids), len(test_ids)]
})
split_table['Percent'] = 100 * split_table['Patients'] / len(features)
print(split_table.round(1).to_string(index=False))
assert set(train_ids).isdisjoint(val_ids)
assert set(dev_ids).isdisjoint(test_ids)
assert len(train_ids) + len(val_ids) + len(test_ids) == len(features)

       Set  Patients  Percent
  Training       264     59.7
Validation        89     20.1
      Test        89     20.1


In [4]:
fig = make_subplots(rows=1, cols=2, subplot_titles=[
    'One dataset, three jobs', 'Observed training and validation patients'
])
for name, count, color in zip(
    split_table['Set'], split_table['Patients'],
    [TRAIN_COLOR, VAL_COLOR, TEST_COLOR]
):
    fig.add_trace(go.Bar(
        x=[count], y=['Patients'], orientation='h', name=name,
        marker_color=color, text=[f'{name}: {count}'], textposition='inside',
        hovertemplate=f'{name}: %{{x}} patients<extra></extra>', showlegend=False
    ), row=1, col=1)
fig.update_layout(barmode='stack')
add_patient_points(fig, row=1, col=2, show_legend=True)
fig.update_xaxes(title_text='Number of patients', row=1, col=1)
fig.update_xaxes(title_text='BMI', row=1, col=2)
fig.update_yaxes(title_text='Progression score', row=1, col=2)
show_chart(fig, 'Real data: split first, then develop the model')

<Interactive Plotly figure: rerun cell in Colab if hidden>

**Notice:** people with similar BMI can have very different targets. A useful model should learn a broad relationship rather than try to explain every difference using BMI alone. The scatterplot omits the final test records.

## Split variants
| Situation | Strategy | Why |
|---|---|---|
| Independent, similarly distributed examples | Random split | A useful first baseline |
| Classification with uneven class sizes | Stratified split | Helps preserve class proportions |
| Repeated records per patient/device | Group-aware split | Keeps related records together |
| Forecasting later outcomes | Time-based split | Trains on the past, evaluates on later cases |

A two-way training/test split can work when all choices are fixed in advance. Repeated tuning requires validation or CV within the development data.

### Real classification example: stratification
We load another bundled real dataset rather than invent labels. This example demonstrates split membership only; it does not train a diagnostic model.

In [5]:
cancer = load_breast_cancer(as_frame=True)
class_X, class_y = cancer.data, cancer.target

# Split the features and labels together so their rows stay aligned.
class_X_train, class_X_holdout, class_y_train, class_y_holdout = train_test_split(
    class_X, class_y,
    test_size=0.25, stratify=class_y, random_state=SEED
)

shares = pd.DataFrame({
    'All rows': class_y.value_counts(normalize=True),
    'Training': class_y_train.value_counts(normalize=True),
    'Holdout': class_y_holdout.value_counts(normalize=True),
}).sort_index()
shares.index = cancer.target_names
print('Observed class proportions:')
print(shares.round(3).to_string())

Observed class proportions:
           All rows  Training  Holdout
malignant     0.373     0.373    0.371
benign        0.627     0.627    0.629


### Quick check
You try ten settings and choose the one with the best test score. Is the test set still independent?

<details><summary>Answer</summary>
No. It influenced selection, so it now acts as validation data. Keep an untouched test set for a final audit.
</details>

**Try it:** change the first split's `test_size` to `0.30`. If the second remains `0.25`, what share is validation?
<details><summary>Answer</summary>
25% of the remaining 70% = 17.5%, before rounding to whole rows. Training is 52.5%.
</details>

# 2. Overfitting and underfitting

We first explore a **decision tree**, then compare **linear regression and polynomial regression** using the same real training and validation patients.

A decision tree asks questions such as “Is BMI below this value?” and predicts the average training target in each final group, called a **leaf**.

**Depth** is the number of question levels. A deeper tree can create more, smaller groups and make a more flexible prediction rule. Its curve looks like a staircase because every input in a leaf gets the same prediction.

- **Underfitting:** the rule is too simple to capture useful structure.
- **Good balance:** captures useful patterns and predicts validation rows relatively well.
- **Overfitting:** fits accidental details; training performance is much better than validation performance.

**Mean squared error (MSE)** averages squared prediction mistakes. Lower is better; large mistakes count more. **RMSE** is its square root, expressed in target-score units.

Here we also show a **baseline** that always predicts the training-set mean. A model should earn its extra complexity by improving on a simple baseline.

In [6]:
# SETTINGS: None means no depth limit.
DEPTHS_TO_SHOW = [1, 3, None]
bmi_grid = np.linspace(X_train.min(), X_train.max(), 250).reshape(-1, 1)

baseline = DummyRegressor(strategy='mean')
baseline.fit(X_train, y_train)
baseline_predictions = baseline.predict(X_val)
print('Mean-baseline validation MSE:',
      round(mean_squared_error(y_val, baseline_predictions), 1))

fig = make_subplots(rows=1, cols=3, shared_yaxes=True,
                    subplot_titles=[f'Depth {depth}' for depth in DEPTHS_TO_SHOW])
fit_results = []
for panel, depth in enumerate(DEPTHS_TO_SHOW, start=1):
    model = DecisionTreeRegressor(max_depth=depth, random_state=SEED)
    model.fit(X_train, y_train)
    train_predictions = model.predict(X_train)
    val_predictions = model.predict(X_val)
    train_mse = mean_squared_error(y_train, train_predictions)
    val_mse = mean_squared_error(y_val, val_predictions)
    fit_results.append({'Depth': str(depth), 'Training MSE': train_mse,
                        'Validation MSE': val_mse, 'Leaves': model.get_n_leaves()})

    add_patient_points(fig, col=panel, show_legend=(panel == 1))
    fig.add_trace(go.Scatter(
        x=bmi_grid[:, 0], y=model.predict(bmi_grid), mode='lines',
        name='Tree prediction', legendgroup='model', showlegend=(panel == 1),
        line=dict(color=MODEL_COLOR, shape='hv')
    ), row=1, col=panel)
    fig.layout.annotations[panel - 1].text += f'<br>Train {train_mse:.0f} | Val {val_mse:.0f}'
fig.update_xaxes(title_text='BMI')
fig.update_yaxes(title_text='Progression score', row=1, col=1)
show_chart(fig, 'Tree flexibility: same patients, different depth', height=500)
print(pd.DataFrame(fit_results).round(1).to_string(index=False))

Mean-baseline validation MSE: 5721.3


<Interactive Plotly figure: rerun cell in Colab if hidden>

Depth  Training MSE  Validation MSE  Leaves
    1        4250.2          4255.4       2
    3        3558.4          4006.0       8
 None        1913.9          7353.0     133


**Read the chart:** the shallow tree makes broad groups. The unlimited tree follows much smaller variations, but its validation performance may get worse. Some patients share the same BMI, so a one-feature tree cannot necessarily achieve zero training error even with unlimited depth.

The best-looking training curve is not necessarily the best predictor. A shallow tree is not automatically underfitting; compare validation errors and the baseline before deciding. With real data, do not expect a perfectly smooth textbook pattern.

## Variant A: sweep through depth values
Use the same rows and metric for every comparison. The final test set stays sealed.

In [7]:
depths = list(range(1, 13))
train_errors, val_errors = [], []
for depth in depths:
    model = DecisionTreeRegressor(max_depth=depth, random_state=SEED)
    model.fit(X_train, y_train)
    train_predictions = model.predict(X_train)
    val_predictions = model.predict(X_val)
    train_errors.append(mean_squared_error(y_train, train_predictions))
    val_errors.append(mean_squared_error(y_val, val_predictions))

best_holdout_depth = depths[int(np.argmin(val_errors))]
fig = go.Figure()
fig.add_trace(go.Scatter(x=depths, y=train_errors, mode='lines+markers',
                        name='Training MSE', line_color=TRAIN_COLOR))
fig.add_trace(go.Scatter(x=depths, y=val_errors, mode='lines+markers',
                        name='Validation MSE', marker_symbol='diamond',
                        line=dict(color=VAL_COLOR, dash='dash')))
fig.add_vline(x=best_holdout_depth, line_dash='dot',
              annotation_text=f'Best validation depth: {best_holdout_depth}')
fig.update_xaxes(title_text='Maximum tree depth', dtick=1)
fig.update_yaxes(title_text='MSE — lower is better')
show_chart(fig, 'Tree depth versus training and validation error')

<Interactive Plotly figure: rerun cell in Colab if hidden>

## Variant B: require more examples in each leaf

### What exactly is a leaf?
Think of a tree as a sequence of sorting questions. Every training patient eventually reaches one **final group**, called a leaf. The prediction for that group is the **average target of the training patients in it**.

With BMI as the only input, each leaf covers an interval of BMI values. A new patient is routed to an interval and gets that group's average progression score. The model does not look at that patient's actual target.

### What does `min_samples_leaf` change?
It sets the **minimum number of training rows allowed in a final group**. The tree rejects a proposed split if it would create a child group smaller than that minimum.

| Setting | What the tree is allowed to do | Likely effect |
|---|---|---|
| `min_samples_leaf=1` | Create a group with just one training patient | Many small steps; can chase individual variation |
| `min_samples_leaf=5` | Keep at least 5 training rows in every leaf | Fewer very small groups |
| `min_samples_leaf=40` | Keep at least 40 training rows in every leaf | Fewer, broader groups; can miss useful detail |

The minimum is **not** the exact size of every leaf, and it does not count validation patients. Repeated BMI values or other stopping conditions can create leaves larger than the minimum. With only one feature, patients with identical BMI cannot be separated by this tree.

**Analogy:** estimating a neighborhood's typical price from one sale is fragile. Averaging several sales can make the estimate steadier, but combining very different neighborhoods can hide important differences.

This is **regularization**: restrict flexibility to reduce the risk of overfitting. Larger leaves often increase training error; validation error tells us whether the restriction helps.

In [8]:
# SETTINGS: unlimited depth isolates the effect of minimum leaf size.
LEAF_SIZES = [1, 5, 15, 40]
leaf_models = {}
leaf_results = []
for leaf_size in LEAF_SIZES:
    model = DecisionTreeRegressor(min_samples_leaf=leaf_size, random_state=SEED)
    model.fit(X_train, y_train)
    leaf_models[leaf_size] = model
    # apply() tells us the final group ID for each training row.
    leaf_ids = model.apply(X_train)
    unique_leaves, counts = np.unique(leaf_ids, return_counts=True)
    assert counts.min() >= leaf_size
    train_predictions = model.predict(X_train)
    val_predictions = model.predict(X_val)
    leaf_results.append({
        'Minimum leaf size': leaf_size, 'Actual smallest leaf': counts.min(),
        'Leaves': model.get_n_leaves(),
        'Training MSE': mean_squared_error(y_train, train_predictions),
        'Validation MSE': mean_squared_error(y_val, val_predictions)
    })
leaf_table = pd.DataFrame(leaf_results)
print(leaf_table.round(1).to_string(index=False))

fig = make_subplots(rows=2, cols=2, shared_xaxes=True, shared_yaxes=True,
                    vertical_spacing=0.17,
                    subplot_titles=[f'Minimum {n} patients per leaf' for n in LEAF_SIZES])
for panel, leaf_size in enumerate(LEAF_SIZES):
    row, col = panel // 2 + 1, panel % 2 + 1
    model = leaf_models[leaf_size]
    result = leaf_results[panel]
    add_patient_points(fig, row=row, col=col, show_legend=(panel == 0))
    fig.add_trace(go.Scatter(
        x=bmi_grid[:, 0], y=model.predict(bmi_grid), mode='lines',
        name='Leaf-average prediction', legendgroup='model', showlegend=(panel == 0),
        line=dict(color=MODEL_COLOR, shape='hv')
    ), row=row, col=col)
    fig.layout.annotations[panel].text += (
        f'<br>{model.get_n_leaves()} leaves | Train {result["Training MSE"]:.0f}'
        f' | Val {result["Validation MSE"]:.0f}'
    )
fig.update_xaxes(title_text='BMI')
fig.update_yaxes(title_text='Progression score')
show_chart(fig, 'Larger minimum leaf sizes restrict the tree', height=800)

 Minimum leaf size  Actual smallest leaf  Leaves  Training MSE  Validation MSE
                 1                     1     133        1913.9          7353.0
                 5                     5      39        2924.5          5735.7
                15                    15      13        3483.3          4250.5
                40                    40       5        3966.7          3689.1


<Interactive Plotly figure: rerun cell in Colab if hidden>

**Read Chart 1:** the dots stay exactly the same; only the minimum group size changes. With a small minimum, the staircase has many small jumps. With a larger minimum, more patients share each prediction. Compare the printed validation scores rather than assuming the smoothest curve always wins.

### Chart 2: open the final groups and count their training patients
Each bar below represents **one actual leaf**, ordered from lower to higher BMI. Its height is the number of training rows in that group. The dashed line is the required minimum. This makes the difference between **minimum leaf size** and **number of leaves** visible.

In [9]:
fig = make_subplots(rows=1, cols=2, shared_yaxes=True,
                    subplot_titles=['Small minimum: many small groups',
                                    'Large minimum: fewer broad groups'])
for panel, leaf_size in enumerate([LEAF_SIZES[0], LEAF_SIZES[-1]], start=1):
    model = leaf_models[leaf_size]
    leaf_ids = model.apply(X_train)
    leaf_rows = []
    # Build one simple dictionary for each actual group.
    for leaf_id in np.unique(leaf_ids):
        in_leaf = (leaf_ids == leaf_id)
        leaf_rows.append({
            'Patients': int(in_leaf.sum()),
            'Mean BMI': X_train[in_leaf, 0].mean(),
            'Prediction': y_train[in_leaf].mean()
        })
    leaf_summary = pd.DataFrame(leaf_rows).sort_values('Mean BMI')
    positions = np.arange(1, len(leaf_summary) + 1)
    fig.add_trace(go.Bar(
        x=positions, y=leaf_summary['Patients'], marker_color=TRAIN_COLOR,
        name=f'Minimum {leaf_size}', showlegend=False,
        customdata=leaf_summary[['Mean BMI', 'Prediction']].to_numpy(),
        hovertemplate='Leaf %{x}<br>Training rows: %{y}<br>Mean BMI: %{customdata[0]:.1f}'
                      '<br>Prediction: %{customdata[1]:.1f}<extra></extra>'
    ), row=1, col=panel)
    fig.add_hline(y=leaf_size, line_dash='dash', row=1, col=panel,
                  annotation_text=f'Required minimum: {leaf_size}')
    fig.layout.annotations[panel-1].text = (
        f'Minimum {leaf_size}: {len(leaf_summary)} actual leaves'
    )
fig.update_xaxes(title_text='Leaf number, ordered by BMI')
fig.update_yaxes(title_text='Training patients in the leaf', row=1, col=1)
show_chart(fig, 'One bar = one final group; hover to inspect its prediction')

example_leaf = leaf_summary.iloc[0]
print('First group in the right-hand chart:')
print('Training patients:', int(example_leaf['Patients']))
print('Group prediction:', round(example_leaf['Prediction'], 1))
print('The required minimum is not the exact size of every group.')

<Interactive Plotly figure: rerun cell in Colab if hidden>

First group in the right-hand chart:
Training patients: 49
Group prediction: 98.9
The required minimum is not the exact size of every group.


**Try it:** replace `40` with `80` in `LEAF_SIZES` and rerun these leaf-size cells. Does the tree create fewer leaves? Does validation error improve or get worse?

**Quick check:** if `min_samples_leaf=15`, does a leaf need exactly 15 validation patients?
<details><summary>Answer</summary>
No. It needs at least 15 training rows. Validation targets never determine leaf boundaries or leaf averages. The number of validation patients routed into a leaf can be different, including zero.
</details>

## Linear regression: a second way to control flexibility

Ordinary **linear regression** fits one straight line through BMI and progression score. It adjusts an intercept (starting height) and a slope to reduce squared training mistakes. Unlike a tree, it does not make separate patient groups.

First, fit the straight-line model directly. A coefficient here describes an association in the training sample; it is not a causal effect of changing BMI.

In [10]:
linear_model = LinearRegression()
linear_model.fit(X_train, y_train)
print(f'Intercept: {linear_model.intercept_:.2f}')
print(f'Slope: {linear_model.coef_[0]:.2f} progression-score units per BMI unit')
print('Training MSE:', round(mean_squared_error(y_train, linear_model.predict(X_train)), 1))
print('Validation MSE:', round(mean_squared_error(y_val, linear_model.predict(X_val)), 1))

Intercept: -134.50
Slope: 10.73 progression-score units per BMI unit
Training MSE: 4012.7
Validation MSE: 3430.1


### Straight line versus increasingly flexible regression curves
A **polynomial regression** model adds powers of BMI as inputs. Degree 1 gives the ordinary straight line. Higher degrees allow curves with more bends. We still use `LinearRegression`: it is called linear because it is linear in its adjustable coefficients, even when its input features include powers.

To compare flexibility just as we did for tree depth, we fit degrees **1, 3, and 12** using the **same training and validation patients**. We scale BMI **inside a pipeline before creating powers** to improve numerical conditioning. The scaler is fitted only on training rows.

A straight line may already suit this noisy relationship. Do not automatically label degree 1 as underfitting or degree 3 as a good fit; let validation results guide the conclusion.

In [11]:
def make_polynomial_regression(degree):
    return make_pipeline(
        StandardScaler(),
        PolynomialFeatures(degree=degree, include_bias=False),
        LinearRegression()
    )

# SETTINGS: degree 1 is ordinary straight-line regression.
REGRESSION_DEGREES = [1, 3, 12]
regression_results = []
fig = make_subplots(rows=1, cols=3, shared_yaxes=True,
                    subplot_titles=[f'Degree {n}' for n in REGRESSION_DEGREES])
for panel, degree in enumerate(REGRESSION_DEGREES, start=1):
    model = make_polynomial_regression(degree)
    model.fit(X_train, y_train)
    train_predictions = model.predict(X_train)
    val_predictions = model.predict(X_val)
    train_mse = mean_squared_error(y_train, train_predictions)
    val_mse = mean_squared_error(y_val, val_predictions)
    regression_results.append({'Degree': degree, 'Training MSE': train_mse,
                               'Validation MSE': val_mse})
    add_patient_points(fig, col=panel, show_legend=(panel == 1))
    fig.add_trace(go.Scatter(
        x=bmi_grid[:, 0], y=model.predict(bmi_grid), mode='lines',
        name='Regression prediction', legendgroup='model', showlegend=(panel == 1),
        line_color=MODEL_COLOR
    ), row=1, col=panel)
    fig.layout.annotations[panel-1].text += f'<br>Train {train_mse:.0f} | Val {val_mse:.0f}'
fig.update_xaxes(title_text='BMI')
fig.update_yaxes(title_text='Progression score', row=1, col=1)
# Start with a readable range; zoom out to inspect extreme predictions.
fig.update_yaxes(range=[0, 380])
show_chart(fig, 'Regression: straight line, moderate curve, flexible curve', height=500)
print(pd.DataFrame(regression_results).round(1).to_string(index=False))

degree_one_model = make_polynomial_regression(1)
degree_one_model.fit(X_train, y_train)
assert np.allclose(degree_one_model.predict(X_val), linear_model.predict(X_val))
print('Check passed: degree 1 matches ordinary linear regression.')

<Interactive Plotly figure: rerun cell in Colab if hidden>

 Degree  Training MSE  Validation MSE
      1        4012.7          3430.1
      3        3982.2          3540.8
     12        3787.6          3748.4
Check passed: degree 1 matches ordinary linear regression.


**Read the regression panels:** complexity changes the prediction shape, not the underlying data split. High degrees may wiggle sharply near the edges or make extreme predictions where data are sparse. The initial vertical range can hide parts of a curve; zoom out to inspect them; the reported errors include all predictions, even outside the visible range.

On a particular split, validation error can be lower than training error because the two sets contain different patients and different levels of difficulty. This alone does not indicate leakage. If the straight line wins, that is a useful result: extra bends were not justified by these validation patients.

### Complexity-versus-error chart for regression
This matches the tree-depth sweep, but the horizontal axis is **polynomial degree**. The vertical axis is logarithmic so very large validation errors remain visible. We use validation to compare these examples; the final test patients stay sealed.

In [12]:
polynomial_degrees = list(range(1, 13))
regression_train_errors, regression_val_errors = [], []
for degree in polynomial_degrees:
    model = make_polynomial_regression(degree)
    model.fit(X_train, y_train)
    train_predictions = model.predict(X_train)
    val_predictions = model.predict(X_val)
    regression_train_errors.append(mean_squared_error(y_train, train_predictions))
    regression_val_errors.append(mean_squared_error(y_val, val_predictions))

best_regression_degree = polynomial_degrees[int(np.argmin(regression_val_errors))]
fig = go.Figure()
fig.add_trace(go.Scatter(x=polynomial_degrees, y=regression_train_errors,
                        mode='lines+markers', name='Training MSE', line_color=TRAIN_COLOR))
fig.add_trace(go.Scatter(x=polynomial_degrees, y=regression_val_errors,
                        mode='lines+markers', name='Validation MSE', marker_symbol='diamond',
                        line=dict(color=VAL_COLOR, dash='dash')))
fig.add_vline(x=best_regression_degree, line_dash='dot',
              annotation_text=f'Best validation degree: {best_regression_degree}')
fig.update_xaxes(title_text='Polynomial degree', dtick=1)
fig.update_yaxes(title_text='MSE — logarithmic scale; lower is better', type='log')
show_chart(fig, 'Regression flexibility versus training and validation error')

<Interactive Plotly figure: rerun cell in Colab if hidden>

### Compare the two model families
| Model | How it predicts | Main flexibility controls |
|---|---|---|
| Decision tree | Average training target inside a learned group | Tree depth; minimum rows per leaf |
| Ordinary linear regression | One straight-line relationship | Which inputs are included |
| Polynomial regression | A curve built from powers of the input | Polynomial degree; optional regularization |

**Try it:** fit a degree-1 regression on only the first 60 training rows, then a degree-12 regression on those same rows. Keep validation unchanged. Which training score looks better? Which validation score is better?

**Quick check:** does a straight line always underfit?
<details><summary>Answer</summary>
No. It can be a useful fit when a broad relationship is close to linear or the data do not support a more complex model. Low training error alone does not justify adding bends.
</details>

### Try it
Repeat the depth sweep using only the first 60 training rows. Keep the validation set unchanged. Does the best depth or training–validation gap change?

### Quick check
A tree has lower training MSE but much higher validation MSE than a simpler tree. Which is preferable, all else equal?
<details><summary>Answer</summary>
The simpler tree with lower validation MSE. Memorization is not the goal. The gap is a warning; also check representativeness and sampling uncertainty.
</details>

# 3. Bias–variance tradeoff

- **Bias:** systematic error from a rule that misses important structure. Think “everyone wears the same shoe size.”
- **Variance:** sensitivity to which examples are used for training. A new sample makes the rule change a lot.
- **Irreducible uncertainty:** variation unexplained by the available inputs, including measurement noise and omitted information.

Simple models often have more bias and less variance. Flexible models often have less bias but more variance. These are tendencies, not guarantees.

## A real-data experiment: resample the training patients
A **bootstrap sample** draws existing training rows **with replacement**. Some patients appear more than once; others are left out of that sample. We train a fresh tree for each sample and compare its predictions.

**We do not create patients, targets, or measurement noise.** The validation rows stay fixed and never enter a bootstrap training sample.

Unlike a simulation, we do not know the true expected progression at each BMI. We can estimate prediction variability, but cannot honestly plot true squared bias or a known noise floor. High error from a stable model is only a clue about bias or missing information, not a direct bias measurement.

In [13]:
# OPTIONAL CODE DETAIL: bootstrap resampling; inspect its charts first.
# SETTINGS: only existing training patients are sampled, with replacement.
BOOTSTRAP_SIZE = 150
N_REPEATS = 40
DEPTHS = list(range(1, 11))
bootstrap_rng = np.random.default_rng(SEED)
bootstrap_indices = []
for repeat in range(N_REPEATS):
    indices = bootstrap_rng.choice(len(X_train), size=BOOTSTRAP_SIZE, replace=True)
    bootstrap_indices.append(indices)

predictions_by_depth = {}
stability_rows = []
for depth in DEPTHS:
    grid_predictions, validation_predictions = [], []
    train_mses, val_mses = [], []
    for indices in bootstrap_indices:
        x_boot, y_boot = X_train[indices], y_train[indices]
        tree = DecisionTreeRegressor(max_depth=depth, random_state=SEED)
        tree.fit(x_boot, y_boot)
        grid_predictions.append(tree.predict(bmi_grid))
        val_prediction = tree.predict(X_val)
        validation_predictions.append(val_prediction)
        train_mses.append(mean_squared_error(y_boot, tree.predict(x_boot)))
        val_mses.append(mean_squared_error(y_val, val_prediction))
    predictions_by_depth[depth] = np.array(grid_predictions)
    validation_predictions = np.array(validation_predictions)
    stability_rows.append({
        'Depth': depth,
        'Mean bootstrap training MSE': np.mean(train_mses),
        'Mean validation MSE': np.mean(val_mses),
        'Prediction variance': np.var(validation_predictions, axis=0).mean(),
    })
stability = pd.DataFrame(stability_rows)

fig = make_subplots(rows=1, cols=3, shared_yaxes=True,
                    subplot_titles=['Depth 1', 'Depth 3', 'Depth 10'])
for panel, depth in enumerate([1, 3, 10], start=1):
    predictions = predictions_by_depth[depth]
    for repeat, prediction in enumerate(predictions[:15]):
        fig.add_trace(go.Scatter(
            x=bmi_grid[:, 0], y=prediction, mode='lines',
            name=f'Resampled model {repeat+1}', showlegend=False,
            line=dict(color=MODEL_COLOR, width=1, shape='hv'), opacity=0.12
        ), row=1, col=panel)
    fig.add_trace(go.Scatter(
        x=bmi_grid[:, 0], y=predictions.mean(axis=0), mode='lines',
        name='Average prediction', legendgroup='average', showlegend=(panel == 1),
        line=dict(color=MODEL_COLOR, width=3)
    ), row=1, col=panel)
    fig.add_trace(go.Scatter(
        x=X_val[:, 0], y=y_val, mode='markers',
        name='Fixed validation patients', legendgroup='validation', showlegend=(panel == 1),
        marker=dict(color=VAL_COLOR, symbol='diamond', size=5, opacity=0.35)
    ), row=1, col=panel)
fig.update_xaxes(title_text='BMI')
fig.update_yaxes(title_text='Predicted progression', row=1, col=1)
show_chart(fig, 'New sample, same settings: do the predictions change?', height=500)

<Interactive Plotly figure: rerun cell in Colab if hidden>

In [14]:
fig = make_subplots(rows=1, cols=2, subplot_titles=[
    'Generalization across resampled training sets', 'Sensitivity to the training sample'
])
fig.add_trace(go.Scatter(
    x=stability['Depth'], y=stability['Mean bootstrap training MSE'],
    mode='lines+markers', name='Mean bootstrap training MSE', line_color=TRAIN_COLOR
), row=1, col=1)
fig.add_trace(go.Scatter(
    x=stability['Depth'], y=stability['Mean validation MSE'],
    mode='lines+markers', name='Mean validation MSE', marker_symbol='diamond',
    line=dict(color=VAL_COLOR, dash='dash')
), row=1, col=1)
fig.add_trace(go.Scatter(
    x=stability['Depth'], y=stability['Prediction variance'],
    mode='lines+markers', name='Prediction variance', line_color=TEST_COLOR
), row=1, col=2)
fig.update_xaxes(title_text='Maximum tree depth', dtick=1)
fig.update_yaxes(title_text='MSE', row=1, col=1)
fig.update_yaxes(title_text='Mean prediction variance (score²)', row=1, col=2)
show_chart(fig, 'Flexibility can reduce training error while increasing sensitivity')
print(stability.round(1).to_string(index=False))

<Interactive Plotly figure: rerun cell in Colab if hidden>

 Depth  Mean bootstrap training MSE  Mean validation MSE  Prediction variance
     1                       3931.1               4379.0                330.9
     2                       3334.1               4067.5                290.9
     3                       3034.9               4559.0                584.5
     4                       2620.9               4953.2                878.8
     5                       2220.7               5549.5               1205.7
     6                       1854.6               6059.3               1533.9
     7                       1542.6               6573.8               1785.1
     8                       1321.0               7030.9               2067.7
     9                       1187.3               7277.3               2184.4
    10                       1096.7               7442.9               2275.8


**What to notice:** deeper trees can reduce training error while becoming less stable across samples. If validation error also worsens, the extra flexibility is not helping generalization. Real curves can be bumpy; changing sample size can change the outcome.

The right chart estimates sensitivity to bootstrap resampling of this particular training dataset. It is not the exact variance over every possible population sample, and neither chart directly measures true bias.

## A simpler second example: linear regression on resampled patients

**Question:** if we select different training patients, will we learn the same straight line?

Imagine asking different groups of people to estimate the relationship between BMI and progression. Each group sees different examples. Their fitted lines may have slightly different slopes and starting heights.

We use **exactly the same real training samples** as the tree experiment above. `bootstrap_indices` is simply a list of row selections already created there. Every selection contains training patients only; the validation patients and final test patients never enter fitting.

### Step 1: fit a straight line repeatedly
1. Pick one of the stored training-row selections.
2. Create a fresh `LinearRegression` model and fit it to those patients.
3. Predict at the same BMI values as every other model.
4. Repeat for the other selections and compare the lines.

The code uses the familiar **create → fit → predict** pattern. The chart formatting below the loop only displays the results.

In [15]:
# Every sample contains existing training patients, selected with replacement.
linear_sample_predictions = []
for sample_ids in bootstrap_indices:
    sample_X = X_train[sample_ids]
    sample_y = y_train[sample_ids]

    model = LinearRegression()          # Create a fresh straight-line model.
    model.fit(sample_X, sample_y)        # Learn from this selection of patients.
    predictions = model.predict(bmi_grid)  # Predict at the same BMI values.
    linear_sample_predictions.append(predictions)

linear_sample_predictions = np.array(linear_sample_predictions)
average_line = linear_sample_predictions.mean(axis=0)

fig = go.Figure()
# Show 12 individual lines so the plot remains easy to read.
for number, predictions in enumerate(linear_sample_predictions[:12], start=1):
    fig.add_trace(go.Scatter(
        x=bmi_grid[:, 0], y=predictions, mode='lines',
        name=f'Training sample {number}', showlegend=False,
        line=dict(color=TRAIN_COLOR, width=1), opacity=0.35
    ))
fig.add_trace(go.Scatter(
    x=bmi_grid[:, 0], y=average_line, mode='lines',
    name='Average across all fitted lines', line=dict(color=MODEL_COLOR, width=4)
))
fig.add_trace(go.Scatter(
    x=X_val[:, 0], y=y_val, mode='markers', name='Fixed validation patients',
    marker=dict(color=VAL_COLOR, symbol='diamond', size=6, opacity=0.4)
))
fig.update_xaxes(title_text='BMI')
fig.update_yaxes(title_text='Predicted progression score')
show_chart(fig, 'Same linear-regression method, different training patients')

<Interactive Plotly figure: rerun cell in Colab if hidden>

**How to read this chart:**
- Each thin teal line is fitted to a different selection of training patients.
- The thick dark line is their average prediction, not a known true relationship.
- Lines close together at the same BMI indicate relatively stable predictions there.
- Wider separation means more sensitivity to which patients were selected.
- Orange diamonds are the same held-out validation patients throughout.

**Connect to variance:** predictions can change even though the algorithm and settings stay the same. That change is caused by the training sample. A straight line may be relatively stable, especially near well-represented BMI values.

**Connect to bias:** stability alone does not establish accuracy. A stable rule can consistently miss useful structure. We still need validation error. Because the real underlying function is unknown, we cannot label the gap between the average line and individual patient outcomes as true bias: those outcomes also reflect other factors and measurement variation.

### Step 2: compare three levels of flexibility
Use the same row selections and validation patients for all three approaches:

| Approach | Prediction rule | What it helps us discuss |
|---|---|---|
| Mean only | Predict the training-sample average for every BMI | A very rigid rule that ignores BMI; possible underfitting |
| Straight line | Fit ordinary linear regression using BMI | A simple relationship that may balance fit and stability |
| Flexible curve | Fit linear regression on polynomial features, degree 8 | Extra flexibility that may become sensitive to the sample |

The curved model still uses `LinearRegression`; polynomial features were introduced in Section 2. More flexibility is not automatically better, and a straight line is not automatically underfitting.

In the code below, `model_name` determines which model to create. Everything after that follows the same fitting and scoring steps. The nested loops mean: **for each model type, repeat over the same patient selections**.

In [16]:
# SETTINGS: try degree 5 or 10, then compare validation error and stability.
FLEXIBLE_DEGREE = 8
regression_names = ['Mean only', 'Straight line', 'Flexible curve']
regression_bootstrap_results = []
fig = make_subplots(rows=1, cols=3, shared_yaxes=True,
                    subplot_titles=regression_names)

for panel, model_name in enumerate(regression_names, start=1):
    sample_grid_predictions = []
    sample_val_predictions = []
    sample_train_errors = []
    sample_val_errors = []

    for sample_ids in bootstrap_indices:
        sample_X = X_train[sample_ids]
        sample_y = y_train[sample_ids]

        if model_name == 'Mean only':
            model = DummyRegressor(strategy='mean')
        elif model_name == 'Straight line':
            model = LinearRegression()
        else:
            model = make_polynomial_regression(FLEXIBLE_DEGREE)

        model.fit(sample_X, sample_y)
        train_predictions = model.predict(sample_X)
        val_predictions = model.predict(X_val)
        sample_grid_predictions.append(model.predict(bmi_grid))
        sample_val_predictions.append(val_predictions)
        sample_train_errors.append(mean_squared_error(sample_y, train_predictions))
        sample_val_errors.append(mean_squared_error(y_val, val_predictions))

    sample_grid_predictions = np.array(sample_grid_predictions)
    sample_val_predictions = np.array(sample_val_predictions)
    mean_train_error = np.mean(sample_train_errors)
    mean_val_error = np.mean(sample_val_errors)
    # At each fixed validation BMI, measure variation across fitted models.
    # Then average that variation across the validation patients.
    prediction_variance = np.var(sample_val_predictions, axis=0).mean()
    regression_bootstrap_results.append({
        'Model': model_name,
        'Mean training MSE': mean_train_error,
        'Mean validation MSE': mean_val_error,
        'Prediction variance': prediction_variance
    })

    for predictions in sample_grid_predictions[:12]:
        fig.add_trace(go.Scatter(
            x=bmi_grid[:, 0], y=predictions, mode='lines',
            showlegend=False, name=model_name + ': one sample',
            line=dict(color=TRAIN_COLOR, width=1), opacity=0.25
        ), row=1, col=panel)
    fig.add_trace(go.Scatter(
        x=bmi_grid[:, 0], y=sample_grid_predictions.mean(axis=0), mode='lines',
        name='Average prediction', legendgroup='average', showlegend=(panel == 1),
        line=dict(color=MODEL_COLOR, width=3)
    ), row=1, col=panel)
    fig.add_trace(go.Scatter(
        x=X_val[:, 0], y=y_val, mode='markers',
        name='Fixed validation patients', legendgroup='validation', showlegend=(panel == 1),
        marker=dict(color=VAL_COLOR, symbol='diamond', size=5, opacity=0.3)
    ), row=1, col=panel)
    fig.layout.annotations[panel-1].text += f'<br>Mean validation MSE: {mean_val_error:.0f}'

fig.update_xaxes(title_text='BMI')
fig.update_yaxes(title_text='Predicted progression score', row=1, col=1)
# Start with a common readable range; zoom out to inspect extreme curves.
fig.update_yaxes(range=[0, 380])
show_chart(fig, 'Rigid rule, straight line, flexible curve: compare stability and error', height=520)
regression_bootstrap_table = pd.DataFrame(regression_bootstrap_results)
print(regression_bootstrap_table.round(1).to_string(index=False))

<Interactive Plotly figure: rerun cell in Colab if hidden>

         Model  Mean training MSE  Mean validation MSE  Prediction variance
     Mean only             5853.4               5846.4                 29.9
 Straight line             3793.0               3529.2                 54.8
Flexible curve             3517.4               5509.0               1052.6


### What should students take away?

Read the plots and the table together:
- **Mean only:** similar-looking horizontal lines can still miss the relationship between BMI and outcome. Check its validation error against the straight line.
- **Straight line:** inspect whether its predictions stay relatively close across samples while improving validation error over the constant baseline.
- **Flexible curve:** inspect whether added bends reduce training error but increase prediction variability or validation error, especially at sparse edges.

The table's **prediction variance** measures how much predictions change across fitted models at the same held-out inputs; its units are progression score squared. It is not the variance of the patients' outcomes or the variance of fold scores. All reported errors include predictions outside the chart's initial vertical range.

**Do not expect a textbook ranking on every real dataset.** Lower variance alone is not enough, and extra flexibility does not guarantee lower bias or better validation performance. This experiment estimates sensitivity to bootstrap sampling of the available training data; it cannot directly measure true bias or the population noise floor.

**Simple lecture message:** “A model should be flexible enough to learn a useful relationship, but stable enough that a different training sample does not change its predictions excessively.”

**Try it:** change `FLEXIBLE_DEGREE` from 8 to 3 and rerun the comparison. Predict first: will the curves tend to become steadier? Does validation performance improve? Keep the final test patients sealed.

## Variant: average several models
Averaging different fitted models can reduce prediction variability. This is the intuition behind **bagging**. Below we compare one full-training tree with the mean prediction of bootstrapped trees. Training sizes differ, so this is a demonstration of an averaging workflow, not an isolated experiment proving an averaging effect. Random forests add further randomness, including feature selection.

In [17]:
single_tree = DecisionTreeRegressor(max_depth=10, random_state=SEED)
single_tree.fit(X_train, y_train)
single_predictions = single_tree.predict(X_val)

bootstrap_predictions = []
for indices in bootstrap_indices:
    tree = DecisionTreeRegressor(max_depth=10, random_state=SEED)
    tree.fit(X_train[indices], y_train[indices])
    bootstrap_predictions.append(tree.predict(X_val))
averaged_predictions = np.mean(bootstrap_predictions, axis=0)

print('One tree validation MSE:',
      round(mean_squared_error(y_val, single_predictions), 1))
print('Average of bootstrapped trees validation MSE:',
      round(mean_squared_error(y_val, averaged_predictions), 1))
print('Averaging can help; this dataset and split do not guarantee improvement.')

One tree validation MSE: 7283.3
Average of bootstrapped trees validation MSE: 5167.0
Averaging can help; this dataset and split do not guarantee improvement.


| Observation | Useful experiments |
|---|---|
| Training and validation errors both high | Better inputs, a more expressive model, less excessive regularization |
| Training low, validation much higher | Simpler model, larger leaves, more representative training data |
| Predictions change across samples | Regularization, more data, averaging models |

**Try it:** raise `BOOTSTRAP_SIZE` and rerun this section. Does prediction variability decrease? Reusing a fixed dataset cannot replace collecting new, representative observations.

**Quick check:** if a model changes wildly with a new training sample, which concept is involved?
<details><summary>Answer</summary>
Variance. Bias is systematic mismatch of the average prediction; it cannot be directly measured here because the true underlying function is unknown.
</details>

# 4. Cross-validation

One validation split can be lucky or unlucky. **Cross-validation (CV)** repeats the rehearsal with several splits.

In **K-fold CV**:
1. Keep the final test set sealed.
2. Divide development data into K folds.
3. Fit a **fresh** model on K−1 folds and validate on the remaining fold.
4. Rotate until every fold has been held out once.
5. Compare settings using mean validation performance.

For K=5, there are five fits per candidate. Each development example is validated once and trained on in the other four runs.

## Switch from a single validation split to CV
We now combine the earlier training and validation rows into development data and use **all 10 real features**. The test patients are exactly the same sealed patients as before. Previous one-feature results are teaching comparisons, not directly comparable benchmarks for these all-feature models.

In [18]:
X_all = features.to_numpy()
X_dev, y_dev = X_all[dev_ids], y[dev_ids]

# SETTINGS: try 3, 5, or 10 folds.
K = 5
kfold = KFold(n_splits=K, shuffle=True, random_state=SEED)
fold_splits = list(kfold.split(X_dev))

# One row per CV run. Start with all training (0); mark validation rows (1).
roles = np.zeros((K, len(X_dev)), dtype=int)
for run, (train_idx, val_idx) in enumerate(fold_splits):
    roles[run, val_idx] = 1
role_names = np.where(roles == 0, 'Training', 'Validation')
fig = go.Figure(go.Heatmap(
    z=roles, x=np.arange(len(X_dev)), y=np.arange(1, K+1),
    zmin=0, zmax=1,
    colorscale=[[0, TRAIN_COLOR], [0.49, TRAIN_COLOR],
                [0.5, VAL_COLOR], [1, VAL_COLOR]],
    customdata=role_names, showscale=True,
    colorbar=dict(tickvals=[0, 1], ticktext=['Training', 'Validation']),
    hovertemplate='Run %{y}<br>Development row %{x}<br>%{customdata}<extra></extra>'
))
fig.update_xaxes(title_text='Position in development data (not time)')
fig.update_yaxes(title_text='Run', dtick=1, autorange='reversed')
show_chart(fig, 'K-fold CV: each patient gets one validation turn')

<Interactive Plotly figure: rerun cell in Colab if hidden>

## First, do it by hand
`clone` creates an unfitted copy for each fold. Only the fold's training patients influence fitting. Trees do not need feature scaling.

In [19]:
candidate = DecisionTreeRegressor(max_depth=3, min_samples_leaf=5, random_state=SEED)
fold_rows = []
for run, (train_idx, val_idx) in enumerate(fold_splits, start=1):
    model = clone(candidate)
    model.fit(X_dev[train_idx], y_dev[train_idx])
    predictions = model.predict(X_dev[val_idx])
    fold_rows.append({
        'Run': run, 'Training patients': len(train_idx),
        'Validation patients': len(val_idx),
        'Validation MSE': mean_squared_error(y_dev[val_idx], predictions)
    })
fold_table = pd.DataFrame(fold_rows)
print(fold_table.round(1).to_string(index=False))
print('\nMean fold MSE:', round(fold_table['Validation MSE'].mean(), 1))
print('Fold standard deviation:', round(fold_table['Validation MSE'].std(ddof=0), 1))

 Run  Training patients  Validation patients  Validation MSE
   1                282                   71          3574.2
   2                282                   71          4908.5
   3                282                   71          4412.3
   4                283                   70          4042.4
   5                283                   70          4246.7

Mean fold MSE: 4236.8
Fold standard deviation: 437.9


Fold standard deviation describes variation between scores. It is **not** a confidence interval: models share much of their training data. Fold sizes here may differ by one patient; an unweighted mean of fold scores differs slightly from MSE across all held-out predictions combined.

## The shortcut: `cross_val_score`
Scikit-learn's scoring helpers use “higher is better,” so the MSE scorer returns **negative MSE**. Negate it once to recover ordinary positive errors; then lower is better.

In [20]:
negative_scores = cross_val_score(
    candidate, X_dev, y_dev, cv=fold_splits,
    scoring='neg_mean_squared_error'
)
positive_mses = -negative_scores
print('Fold MSEs:', positive_mses.round(1))
print('Mean MSE:', round(positive_mses.mean(), 1))
assert np.allclose(positive_mses, fold_table['Validation MSE'])
print('Check passed: helper scores match the manual loop.')

Fold MSEs: [3574.2 4908.5 4412.3 4042.4 4246.7]
Mean MSE: 4236.8
Check passed: helper scores match the manual loop.


## Optional shortcut: compare settings with GridSearchCV, then refit
`GridSearchCV` compares every combination using the **same folds**. It selects the highest negative MSE, equivalent to the lowest positive MSE. With `refit=True`, it then fits the chosen setup again on **all development patients**.

The best CV score is used for selection and can be optimistic after many comparisons. The final test set is still excluded.

In [21]:
search = GridSearchCV(
    DecisionTreeRegressor(random_state=SEED),
    param_grid={
        'max_depth': [1, 2, 3, 5, None],
        'min_samples_leaf': [1, 5, 15]
    },
    cv=fold_splits, scoring='neg_mean_squared_error', refit=True, n_jobs=1
)
search.fit(X_dev, y_dev)
comparison = pd.DataFrame({
    'Depth': [str(p['max_depth']) for p in search.cv_results_['params']],
    'Minimum leaf size': search.cv_results_['param_min_samples_leaf'],
    'Mean CV MSE': -search.cv_results_['mean_test_score'],
    'Fold score SD': search.cv_results_['std_test_score']
}).sort_values('Mean CV MSE')
print(comparison.head(8).round(1).to_string(index=False))
print('\nChosen parameters:', search.best_params_)
print(f'Best mean CV MSE: {-search.best_score_:.1f}')
print('Comparison fits:', len(search.cv_results_['params']) * K, '+ one final refit')

Depth  Minimum leaf size  Mean CV MSE  Fold score SD
    2                  1       3984.9          404.7
    2                  5       3984.9          404.7
    2                 15       3984.9          404.7
    3                 15       4106.6          271.9
 None                 15       4121.7          317.6
    5                 15       4152.3          276.1
    3                  1       4209.9          452.3
    3                  5       4236.8          437.9

Chosen parameters: {'max_depth': 2, 'min_samples_leaf': 1}
Best mean CV MSE: 3984.9
Comparison fits: 75 + one final refit


## Variant: compare model families and keep preprocessing inside CV
A **Ridge regression** model learns an additive linear rule, with a penalty to discourage extreme coefficients. Its inputs should be scaled. A **pipeline** fits scaling anew within each training fold, then applies that fitted scaling to validation patients. Fitting a scaler on the full dataset before CV would leak information.

We compare a mean baseline, this linear model, and the tuned tree using the same folds. The tree's best score already benefited from parameter selection; it is not an unbiased comparison with the untuned alternatives. Nested CV is an advanced option when you need a selection-adjusted CV evaluation.

In [22]:
ridge_pipeline = make_pipeline(StandardScaler(), Ridge(alpha=10.0))
model_options = {
    'Mean baseline': DummyRegressor(strategy='mean'),
    'Scaled Ridge (fixed alpha=10)': ridge_pipeline,
    'Tree (settings selected by CV)': clone(search.best_estimator_)
}
model_comparison = []
for name, model in model_options.items():
    errors = -cross_val_score(model, X_dev, y_dev, cv=fold_splits,
                              scoring='neg_mean_squared_error')
    model_comparison.append({'Model': name, 'Mean CV MSE': errors.mean()})
model_comparison = pd.DataFrame(model_comparison).sort_values('Mean CV MSE')
print(model_comparison.round(1).to_string(index=False))

# Make the final choice using development CV only, never the final test result.
chosen_name = model_comparison.iloc[0]['Model']
final_model = clone(model_options[chosen_name])
final_model.fit(X_dev, y_dev)
print('\nChosen for final audit:', chosen_name)

                         Model  Mean CV MSE
 Scaled Ridge (fixed alpha=10)       3078.6
Tree (settings selected by CV)       3984.9
                 Mean baseline       6136.7

Chosen for final audit: Scaled Ridge (fixed alpha=10)


## Final audit: reveal the test targets once
All choices are complete. We now predict for the sealed test patients and report the result without using it to tune again.

Rerunning unchanged code is reproducible. Changing settings because of this final result would make the test set part of selection. Its score also says nothing about transfer to a new population unless the split represents that use.

In [23]:
# All model choices are now complete. Reveal the test targets for the audit.
X_test, y_test = X_all[test_ids], y[test_ids]
final_predictions = final_model.predict(X_test)
final_test_mse = mean_squared_error(y_test, final_predictions)
print('Final model:', chosen_name)
print(f'Untouched-test MSE: {final_test_mse:.1f}')
print(f'Untouched-test RMSE: {np.sqrt(final_test_mse):.1f} progression-score units')
print('Test patients:', len(test_ids))

fig = go.Figure()
fig.add_trace(go.Scatter(
    x=y_test, y=final_predictions, mode='markers', name='Held-out test patients',
    marker=dict(color=TEST_COLOR, size=8, opacity=0.7),
    hovertemplate='Observed: %{x:.1f}<br>Predicted: %{y:.1f}<extra></extra>'
))
low = min(y_test.min(), final_predictions.min())
high = max(y_test.max(), final_predictions.max())
fig.add_trace(go.Scatter(x=[low, high], y=[low, high], mode='lines',
                        name='Perfect predictions', line=dict(color='gray', dash='dot')))
fig.update_xaxes(title_text='Observed progression score')
fig.update_yaxes(title_text='Predicted progression score')
show_chart(fig, 'Final audit: closer to the diagonal means a smaller error')

Final model: Scaled Ridge (fixed alpha=10)
Untouched-test MSE: 2875.8
Untouched-test RMSE: 53.6 progression-score units
Test patients: 89


<Interactive Plotly figure: rerun cell in Colab if hidden>

## Other cross-validation variants
| Variant | Use when | Caution |
|---|---|---|
| Shuffled K-fold | Independent regression records | Do not casually shuffle times or related records |
| Stratified K-fold | Independent classification records | Need enough rows per class |
| Group K-fold | Repeated records from the same entity | Use real group IDs |
| Time-series split | Predicting later periods | Sort by real time; consider a gap and feature availability |
| Leave-one-out | Very small datasets | Many fits and potentially unstable estimates |
| Repeated K-fold | Checking sensitivity to fold assignments | More computation; does not fix leakage |

### Real-data stratified K-fold
This demonstrates fold membership using the real classification labels loaded earlier.

In [24]:
stratified = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)
stratified_rows = []
for run, (tr, va) in enumerate(stratified.split(class_X, class_y), start=1):
    stratified_rows.append({
        'Run': run, 'Training rows': len(tr), 'Validation rows': len(va),
        'Validation class-1 share': class_y.iloc[va].mean()
    })
print(pd.DataFrame(stratified_rows).round(3).to_string(index=False))

 Run  Training rows  Validation rows  Validation class-1 share
   1            455              114                     0.623
   2            455              114                     0.623
   3            455              114                     0.632
   4            455              114                     0.632
   5            456              113                     0.628


### Optional extension: reusable group and time templates
These bundled datasets do not supply repeated-visit group IDs or a prediction-time chronology. We should not invent those columns or treat their row order as time.

The functions below are ready for your own real dataset with those fields. They define split indices; they do not fabricate a demonstration dataset. A task can require both grouping and time constraints.

In [25]:
def group_fold_indices(real_features, real_targets, real_group_ids, n_splits=5):
    """Keep each real group entirely on one side of every split."""
    splitter = GroupKFold(n_splits=n_splits)
    splits = list(splitter.split(real_features, real_targets, groups=real_group_ids))
    group_ids = np.asarray(real_group_ids)
    for train_idx, val_idx in splits:
        assert set(group_ids[train_idx]).isdisjoint(set(group_ids[val_idx]))
    return splits


def time_fold_indices(real_table, time_column, n_splits=3, gap=1):
    """Sort by actual time, then create past-to-future splits."""
    ordered = real_table.sort_values(time_column).reset_index(drop=True)
    splitter = TimeSeriesSplit(n_splits=n_splits, gap=gap)
    return ordered, list(splitter.split(ordered))

# With YOUR data, for example:
# group_splits = group_fold_indices(X, y, patient_ids)
# ordered_data, time_splits = time_fold_indices(data, 'visit_date')
# gap counts rows, not days. For irregular dates or ties, define time windows
# carefully; repeated timestamps may need to stay together.
print('Group and time helpers defined. No artificial group IDs or dates were created.')

Group and time helpers defined. No artificial group IDs or dates were created.


## Practical checklist
- Define what is known at prediction time and which future cases matter.
- Choose a meaningful metric before comparing models.
- Match splits to the task: independent rows, groups, classes, or time.
- Put learned preprocessing inside each training fold.
- Compare candidates using the same splits.
- Keep final test data out of model selection.
- Inspect variation, weak subgroups, and representativeness, not only the average.

## Final knowledge check
1. Why is a tiny training error insufficient?
2. What is a sign of high variance in the resampling experiment?
3. Can we measure true bias directly from this real dataset?
4. How often is a row used for validation in one complete 5-fold run?
5. Where should scaling be fitted during CV?
6. What follows model selection by CV?

<details><summary>Reveal answers</summary>

1. It may reflect memorization; we need held-out performance.
2. Predictions change substantially across resampled training sets.
3. No. The underlying true function is unknown; error and stability provide clues.
4. Once; it is in training in the other four runs.
5. Inside each training fold, usually through a pipeline.
6. Refit the chosen setup on development data and audit once on the untouched test set.

</details>

## Dataset and tool references
- [Diabetes dataset: scikit-learn documentation](https://scikit-learn.org/stable/modules/generated/sklearn.datasets.load_diabetes.html). Use `scaled=False` to load raw features; no whole-dataset scaling is learned by this notebook.
- [Wisconsin Diagnostic Breast Cancer dataset: scikit-learn documentation](https://scikit-learn.org/stable/modules/generated/sklearn.datasets.load_breast_cancer.html).
- Original diabetes study: Efron, Hastie, Johnstone, and Tibshirani (2004), *Least Angle Regression*. Dataset attribution is also available through `diabetes.DESCR` and `cancer.DESCR`.

**Remember:** fit on training data, choose with validation or CV, and reserve the test set for the final audit.